# 线性回归的从零开始实现

## 线性模型
$$
\hat{\mathbf{y}} = \mathbf{X}\mathbf{w} + b
$$
其中：

| 符号 | 含义 |
|---|---|
| $\mathbf{X}$ | 特征矩阵 |
| $\mathbf{w}$ | 权重向量 |
| $b$ | 偏置 |
| $\hat{\mathbf{y}}$ | 模型预测值 |

In [1]:
import torch
import matplotlib.pyplot as plt

##  1.生成人造数据集

为了验证线性回归模型，我们先根据一组已知参数生成数据。

真实模型为：

$$
\mathbf{y} = \mathbf{X}\mathbf{w} + b + \epsilon
$$

这里设定：

$$
\mathbf{w} = [2, -3.4]^T,\quad b = 4.2
$$

其中 $\epsilon$ 表示噪声。

In [2]:
def synthetic_data(w, b, num_examples):
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape(-1, 1)
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

## PyTorch 常用函数说明

`torch.tensor()` 用来创建张量。张量可以理解为 PyTorch 里的数组，它既可以表示标量、向量，也可以表示矩阵。比如 `torch.tensor([2.0, -3.4])` 创建了一个一维张量，用来表示线性回归中的真实权重向量。

`torch.normal()` 用来从正态分布中随机生成数据。比如 `torch.normal(0, 1, (1000, 2))` 表示生成一个形状为 `[1000, 2]` 的张量，其中每个元素都来自均值为 0、标准差为 1 的正态分布。在这里，它用来生成 1000 个样本，每个样本有 2 个特征。

`torch.matmul()` 用来做矩阵乘法。在线性回归中，`torch.matmul(X, w)` 对应数学公式中的 $Xw$。如果 `X` 的形状是 `[1000, 2]`，`w` 的形状是 `[2]`，那么相乘后会得到 `[1000]`，表示每个样本对应的预测值。

`reshape()` 用来改变张量的形状，但不会改变张量里面的数值。比如 `y.reshape(-1, 1)` 会把原本形状为 `[1000]` 的一维标签变成 `[1000, 1]` 的列向量。这里的 `-1` 表示让 PyTorch 自动推断这一维的大小。

## 2. 读取小批量数据

训练模型时，我们通常不会一次性把整个数据集都送入模型，而是每次随机取出一小批样本进行训练。
这一小批样本称为一个 batch，batch 中样本的数量称为 batch size。

下面手写一个 `data_iter` 函数，用来每次随机返回一小批特征和标签。

In [3]:
import random
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(
            indices[i: min(i + batch_size, num_examples)]
        )
        yield features[batch_indices], labels[batch_indices]
batch_size = 10
for X, y in data_iter(batch_size, features, labels):
    print(X)
    print(y)
    break

tensor([[-5.3285e-01, -2.8909e-01],
        [-3.7909e-01,  2.9462e+00],
        [-2.2579e-01, -5.5365e-01],
        [ 2.9530e-01, -1.6043e+00],
        [-9.7474e-02, -2.0878e-02],
        [-1.3559e+00,  1.4352e-01],
        [ 7.3861e-01,  2.9278e-01],
        [-1.2891e+00,  1.0865e-01],
        [ 1.8836e-01, -9.5447e-04],
        [ 1.1806e+00,  2.1538e+00]])
tensor([[ 4.1186],
        [-6.5705],
        [ 5.6207],
        [10.2508],
        [ 4.0779],
        [ 1.0162],
        [ 4.6949],
        [ 1.2386],
        [ 4.5659],
        [-0.7466]])


### 代码说明：读取小批量数据

这段代码的作用是手写一个小批量数据读取器 `data_iter`，它每次从完整数据集中随机取出一小批样本，返回对应的特征和标签。

`import random` 导入的是 Python 自带的随机模块，后面会用 `random.shuffle()` 打乱样本顺序。这样做的目的是避免模型每次都按照固定顺序读取数据，从而让训练过程更随机、更接近小批量随机梯度下降的思想。

`data_iter(batch_size, features, labels)` 是一个数据迭代器函数。这里的 `batch_size` 表示每次取多少个样本，`features` 是全部样本的特征矩阵，`labels` 是全部样本对应的标签。

函数内部先用 `len(features)` 得到样本总数。因为 `features` 的每一行表示一个样本，所以 `len(features)` 就等于数据集中的样本数量。

接着，`indices = list(range(num_examples))` 生成所有样本的编号。例如如果有 1000 个样本，那么 `indices` 就是从 `0` 到 `999` 的列表。每个编号都对应数据集中的一个样本。

然后，`random.shuffle(indices)` 会把这些样本编号随机打乱。注意，这里打乱的是索引顺序，而不是直接打乱 `features` 和 `labels` 本身。这样可以保证特征和标签仍然是一一对应的。

下面的 `for i in range(0, num_examples, batch_size)` 表示按照 `batch_size` 的间隔遍历整个数据集。比如 `batch_size = 10`，那么 `i` 会依次取 `0, 10, 20, 30, ...`，每次处理 10 个样本。

`batch_indices = torch.tensor(indices[i: min(i + batch_size, num_examples)])` 的作用是取出当前这一批样本的编号，并把它转换成 PyTorch 张量。这里使用 `min(i + batch_size, num_examples)` 是为了防止最后一批数据越界。比如样本总数不能刚好被 `batch_size` 整除时，最后一批可能不足 10 个样本。

`yield features[batch_indices], labels[batch_indices]` 会根据当前批次的编号，从 `features` 和 `labels` 中取出对应的小批量数据。这里的 `yield` 和 `return` 不一样，`yield` 会让函数变成一个生成器，每次循环只返回一个 batch，而不是一次性返回所有数据。

函数外部，`batch_size = 10` 表示每次读取 10 个样本。后面的 `for X, y in data_iter(batch_size, features, labels)` 会从数据迭代器中取出一个小批量，其中 `X` 是这一批样本的特征，`y` 是这一批样本对应的标签。

最后的 `print(X)` 和 `print(y)` 用来查看这个小批量数据长什么样。`break` 表示只查看第一个 batch，打印一次后就停止循环。如果不写 `break`，它会把整个数据集的所有 batch 都打印出来。

这段代码的核心作用可以概括为：

$$
完整数据集 \rightarrow 打乱索引 \rightarrow 按 batch\ size 切分 \rightarrow 每次返回一小批样本
$$

其中，返回的 `X` 和 `y` 通常满足：

| 变量 | 含义 | 形状 |
|---|---|---|
| `X` | 一个小批量的特征 | `[batch_size, 2]` |
| `y` | 一个小批量的标签 | `[batch_size, 1]` |

在当前代码中，`batch_size = 10`，所以一般情况下：

$$
X \in \mathbb{R}^{10 \times 2}
$$

$$
y \in \mathbb{R}^{10 \times 1}
$$

这一步是后面训练模型的基础，因为小批量随机梯度下降每次都需要从数据集中取出一个 batch，然后用这个 batch 计算损失和梯度。

## 3. 初始化模型参数

线性回归需要学习两个参数：权重 `w` 和偏置 `b`。
因为每个样本有 2 个特征，所以权重 `w` 初始化为形状 `[2, 1]` 的张量。
偏置 `b` 是一个标量。

为了让 PyTorch 能够自动计算梯度，需要设置 `requires_grad=True`。

In [4]:
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
print(w)
print(b)

tensor([[ 8.7653e-05],
        [-7.4572e-03]], requires_grad=True)
tensor([0.], requires_grad=True)


| `torch.zeros()` 创建全 0 张量

## 4. 定义线性回归模型

线性回归模型的数学形式是：

$$
\hat{y} = Xw + b
$$

其中，`X` 是输入特征，`w` 是权重，`b` 是偏置，`\hat{y}` 是模型预测值。

In [5]:
def linreg(X, w, b):
    return torch.matmul(X, w) + b

##  5.定义平方损失函数

线性回归使用平方误差来衡量预测值和真实值之间的差距。
对于一个样本，平方损失可以写成：

$$
l = \frac{1}{2}(\hat{y} - y)^2
$$

其中，$\hat{y}$ 是模型预测值，$y$ 是真实标签。
前面的 $\frac{1}{2}$ 是为了后面求导时抵消平方项带来的系数 2，使梯度形式更简洁。

In [6]:
def squared_loss(y_hat, y):
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

## 6. 手写小批量随机梯度下降

训练模型的目标是不断调整参数，使损失函数逐渐减小。

梯度下降的基本更新公式是：

$$
参数 \leftarrow 参数 - 学习率 \times 梯度
$$

在线性回归中，需要更新的参数是权重 `w` 和偏置 `b`。

In [7]:
def sgd(params, lr, batch_size):
    with torch.no_grad():#暂停梯度追踪
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()#清空梯度，防止梯度累积

## 7. 训练模型

训练过程就是重复执行以下步骤：

1. 从数据集中随机取出一个小批量；
2. 用当前参数计算预测值；
3. 根据预测值和真实标签计算损失；
4. 反向传播，计算参数梯度；
5. 使用 SGD 更新参数；
6. 重复多轮，直到损失足够小。

这里设置学习率为 `0.03`，训练轮数为 `3`。

In [11]:
lr = 0.03
num_epochs = 3
net = linreg
loss = squared_loss

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X, w, b), y)
        l.sum().backward()
        sgd([w, b], lr, batch_size)

    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f"epoch {epoch + 1}, loss {float(train_l.mean()):f}")

epoch 1, loss 0.000047
epoch 2, loss 0.000048
epoch 3, loss 0.000048


## 8. 检查训练结果

因为这个数据集是我们自己生成的，所以真实参数是已知的。

真实权重为：

$$
\mathbf{w} = [2, -3.4]^T
$$

真实偏置为：

$$
b = 4.2
$$

如果模型训练成功，学到的 `w` 和 `b` 应该接近这两个真实值。

In [12]:
print("true_w:", true_w)
print("learned_w:", w.reshape(-1))
print("w error:", true_w - w.reshape(-1))

print("true_b:", true_b)
print("learned_b:", b.item())
print("b error:", true_b - b.item())

true_w: tensor([ 2.0000, -3.4000])
learned_w: tensor([ 2.0002, -3.3992], grad_fn=<ViewBackward0>)
w error: tensor([-0.0002, -0.0008], grad_fn=<SubBackward0>)
true_b: 4.2
learned_b: 4.1998748779296875
b error: 0.00012512207031267764


## 9. 本节总结
本节用 PyTorch 的基础张量操作，手写实现了线性回归的完整训练流程。
完整流程包括：
- 生成人造数据集；
- 手写小批量数据读取器；
- 初始化权重 `w` 和偏置 `b`；
- 定义线性回归模型；
- 定义平方损失函数；
- 手写小批量随机梯度下降；
- 训练模型；
- 检查训练得到的参数是否接近真实参数。
这一节最重要的对应关系是：
$$
\hat{y} = Xw + b
$$
对应代码：
```python
torch.matmul(X, w) + b